## Load data

Import packages and configure root.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys, pathlib, subprocess
from pathlib import Path
!pip install pyfaidx
from pyfaidx import Fasta
from huggingface_hub import hf_hub_download

COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/mini-gLM.git"
repo_dir = Path("mini-gLM")
if COLAB:
    root = Path("/content/mini-gLM")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

## Model architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from src.transformer import MoETransformer

# Set seed
torch.manual_seed(111)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

import torch
import torch.nn as nn
import torch.nn.functional as F

class DenseGLM(nn.Module):
    def __init__(self, vocab_size, num_blocks, d_model, num_heads, h_dim, num_experts, top_k, p_drop):

        self.vocab_size = vocab_size
        self.num_blocks = num_blocks
        self.d_model = d_model
        self.num_heads = num_heads
        self.h_dim = h_dim
        self.num_experts = num_experts
        self.top_k = top_k
        self.p_drop = p_drop

        self.embedding = nn.Embedding(
            num_embeddings = self.vocab_size,
            embedding_dim = self.d_model
        )

        self.moe_transformer = MoETransformer(
            d_model = self.d_model,
            num_heads = self.num_heads,
            h_dim = self.h_dim,
            num_experts = self.num_experts,
            top_k = self.top_k,
            p_drop = self.p_drop
        )

        self.model = nn.ModuleList([
            self.moe_transformer for _ in range(num_blocks)
        ])

        self.final_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):

        x = self.model(x)
        logits = self.final_head(x)

        return logits

## Training

Implement masked language modeling training objective.
- Embedding module
- Relative positional embeddings
- Batching [B, L, D]
    - Token right padding to max length in batch
- Attention mask to ignore padding tokens
- Max sequence length cutoff
- Masked token prediction objective
Reference: https://huggingface.co/learn/llm-course/en/chapter6/5

Custom Dataset and Dataloader.

In [ ]:
from src.tokenize import BPETokenizer
from torch.utils.data import DataLoader, Dataset, BatchSampler
from torch.nn.utils.rnn import pad_sequence


class hg38Data(Dataset):
    """Custom dataset to load hg38 pre-training data from HuggingFace"""
    def __init__(self, data_path, merge_rules_path, token_map_path):

        # Get sequences from HF path to tokenized dataset
        self.merge_rules_path = merge_rules_path
        self.token_map_path = token_map_path

        # Get tokenizer learned params and load them into the tokenizer

        self.tokenizer = BPETokenizer()

        # Load data from HF and tokenize
        self.sequence_list = pd.read_csv(data_path, compression = "gzip", usecols = ["sequence"])["sequence"].to_list()
        self.tokenized_list = self.tokenizer.tokenize(self.sequence_list).sort(key = len)


    def __len__(self):
        return len(self.tokenized_list)

    def __getitem__(self, idx):
        tokenized = self.tokenized_list[idx]
        tokenized = torch.tensor(tokenized)

        return tokenized


class DynamicBatchSampler(BatchSampler):
    """
    Dynamic batching. For each batch, we set a constant batch_space, meaning the length of the longest sequence * # of sequences is constant per batch.
    This way, each batch contains a different dimension, but an approximately equal amount of tokens. This allows us to save attention compute cost by 
    grouping similar length sequences together, reducing the # of padding tokens needed. We also shuffle the order in which the batches are fed 
    to the DataLoader to minimize sequence length bias when training.
    
    Note: this implementation also assumes that the tokenized sequence data is already sorted from shortest to longest.
    """

    def __init__(self, dataset, batch_space):
        self.batch_space = batch_space
        self.dataset = dataset
        self.dataset_size = len(self.dataset)
    
    def __iter__(self):
        
        batches = []
        batch = []

        for i in range(self.dataset_size):
            batch.append(i)

            if len(self.dataset[i]) * len(batch) >= self.batch_space:
                batches.append(batch)
                batch = []
        
        # Include final batch
        if batch:
            batches.append(batch)
        
        np.random.shuffle(batches)

        for batch in batches:
            yield batch

    def __len__(self):
        num_batches = 0
        batch_size = 0
    
        # Iterate through same logic as the iter function
        for i in range(self.dataset_size):
            batch_size += 1

            if len(self.dataset[i]) * batch_size >= self.batch_space:
                num_batches += 1
                batch_size = 0
            
        if batch_size > 0:
            num_batches += 1

        return num_batches



# Collate function to inclue the collation 

class DynamicCollate():
    def __init__(self, vocab_size, masking_prob, randomize_prob, no_change_prob):
        self.vocab_size = vocab_size
        self.padding_token = self.vocab_size + 1
        self.masking_token = self.vocab_size + 2
        self.masking_prob = masking_prob
        self.randomize_prob = randomize_prob
        self.no_change_prob = no_change_prob
    
    def __call__(self, batch):

        # Right padding to [B, L_max], where L_max is the length of the longest sequence in the batch
        padded = pad_sequence(
            sequences = batch, 
            batch_first = True, 
            padding_value = self.padding_token
        )
        B, L = padded.shape

        # Generate the attention mask [B, 1, 1, L]
        attn_mask = (padded == self.padding_token).unsqueeze(-2).unsqueeze(-2)

        # Select 15% of tokens in batch
        
        # Generate random tensor, then select cutoff above specified 0.85
        predict_idx = torch.arange(B, L)

        # 80% masked, 10% mutated, 10% unchanged
        mask_idx = 
        mutate_idx = 
        same_idx = 

        out = padded.copy()
        out[mask_idx] = self.masking_token
        out[mutate_idx] = 


        # This class needs to return 
        # 1) The idx that were selected for prediction
        # 2) The true tokens for those idx
        # 3) The attention mask for padded tokens

https://www.youtube.com/watch?v=JDy58DtZC_g

https://medium.com/@haleema.ramzan/how-to-build-a-custom-batch-sampler-in-pytorch-ce04161583ee

MLM masking.

In [ ]:
def train_glm():

    # Load train and val data
    train_data = hg38Data()
    val_data  = hg38Data()

    train_loader = DataLoader(
        dataset = train_data, 
        batch_sampler = DynamicBatchSampler(),
        collate_fn = dynamic_collate       
    )
    val_loader = DataLoader(
        val_data,
        batch_sampler = DynamicBatchSampler(),
        collate_fn = dynamic_collate   
    )

    

